# Boulder detection with Detectron2

This notebook downloads pretrained Detectron2 model weights from GitHub Releases and applies boulder detection to image data provided as a Dropbox ZIP archive.

- Supported image formats: `.tif`, `.tiff`, `.jpg`, `.jpeg`
- Input archive: ZIP file from Dropbox
- Output: COCO-format JSON for each image
- Edit only the **Settings** cell first.


In [ ]:
# ============================================================
# Settings: edit only this cell first
# ============================================================

# 1) GitHub Release asset URL for the Detectron2 .pth file.
MODEL_URL = "https://github.com/keitaroyamada/BoulderCalculator/releases/download/v1.0.0/model_0016559.pth"

# 2) Dropbox shared URL for a ZIP file containing .tif/.tiff/.jpg/.jpeg images.
# Example: https://www.dropbox.com/scl/fi/xxxx/images.zip?rlkey=yyyy&dl=0
DROPBOX_ZIP_URL = "https://www.dropbox.com/s/XXXX/images.zip?dl=0"

# 3) Working directories in Colab.
PROJECT_DIR = "/content/boulder_detection"
DATA_DIR = f"{PROJECT_DIR}/images"
OUTPUT_DIR = f"{PROJECT_DIR}/outputs"
MODEL_PATH = f"{PROJECT_DIR}/model_final.pth"
ZIP_PATH = f"{PROJECT_DIR}/images.zip"

# 4) Model settings.
NUM_CLASSES = 1
CLASS_NAMES = ["Boulder"]
SCORE_THRESH_TEST = 0.70
DETECTIONS_PER_IMAGE = 500
BASE_CONFIG = "COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"

# 5) Sliding-window settings.
WINDOW_HEIGHT = 2000
WINDOW_WIDTH = 2000
STEP_RATE_Y = 0.25
STEP_RATE_X = 0.25
APPLY_SHARPENING = False
SAVE_ANNOTATED_TILES = False

# 6) Input image extensions.
IMAGE_EXTENSIONS = (".tif", ".tiff", ".jpg", ".jpeg", ".TIF", ".TIFF", ".JPG", ".JPEG")

# 7) Optional: limit the number of images for a quick test. Use None to process all images.
MAX_IMAGES = None

print("Settings loaded.")


## 1. Install libraries

In Colab, select a GPU runtime before running the notebook.

Use `Runtime > Change runtime type > T4 GPU` or a similar GPU option.


In [ ]:
import importlib.util
import sys

# Detectron2 installation can take several minutes on Colab.
# If this cell fails, restart runtime and run again.
if importlib.util.find_spec("detectron2") is None:
    !python -m pip install -q 'git+https://github.com/facebookresearch/detectron2.git'

!python -m pip install -q tifffile tqdm opencv-python-headless pillow

import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

## 2. Download model and image ZIP

Dropbox shared links are automatically converted to direct-download URLs.


In [ ]:
import os
import zipfile
import subprocess
from pathlib import Path
from urllib.parse import urlparse, parse_qs, urlencode, urlunparse

os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

def to_direct_dropbox_url(url: str) -> str:
    """Convert common Dropbox shared URLs to direct-download URLs."""
    if "dropbox.com" not in url:
        return url
    url = url.replace("www.dropbox.com", "dl.dropboxusercontent.com")
    parsed = urlparse(url)
    query = parse_qs(parsed.query)
    query["dl"] = ["1"]
    return urlunparse(parsed._replace(query=urlencode(query, doseq=True)))

def run_download(url: str, out_path: str):
    if not url or "USER/REPO" in url or "XXXX" in url:
        raise ValueError(f"Please set a valid URL first: {url}")
    cmd = ["wget", "-q", "--show-progress", "-O", out_path, url]
    subprocess.run(cmd, check=True)

print("Downloading model...")
run_download(MODEL_URL, MODEL_PATH)
print(f"Model saved to: {MODEL_PATH}")

print("Downloading image ZIP...")
direct_zip_url = to_direct_dropbox_url(DROPBOX_ZIP_URL)
run_download(direct_zip_url, ZIP_PATH)
print(f"ZIP saved to: {ZIP_PATH}")

print("Extracting images...")
with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    zf.extractall(DATA_DIR)
print(f"Images extracted to: {DATA_DIR}")

## 3. Load Detectron2 model

In [ ]:
import os
import math
import json
import glob
import cv2
import numpy as np
from PIL import Image
Image.MAX_IMAGE_PIXELS = None
from tqdm.notebook import tqdm
from tifffile import TiffFile

import detectron2
from detectron2.utils.logger import setup_logger
setup_logger()
from detectron2 import model_zoo
from detectron2.engine import DefaultPredictor
from detectron2.config import get_cfg
from detectron2.data import MetadataCatalog
from detectron2.utils.visualizer import Visualizer

MetadataCatalog.get("boulder_meta").thing_classes = CLASS_NAMES
detector_metadata = MetadataCatalog.get("boulder_meta")

cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file(BASE_CONFIG))
cfg.MODEL.WEIGHTS = MODEL_PATH
cfg.MODEL.ROI_HEADS.NUM_CLASSES = NUM_CLASSES
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = SCORE_THRESH_TEST
cfg.TEST.DETECTIONS_PER_IMAGE = DETECTIONS_PER_IMAGE
cfg.INPUT.MAX_SIZE_TEST = 0
cfg.INPUT.MIN_SIZE_TEST = 0

predictor = DefaultPredictor(cfg)
print("Model loaded.")

## 4. Utility functions

In [ ]:
class CocoJsonInitialize:
    def __init__(self, class_names):
        self.jData = {
            "licenses": [{"name": "", "id": 0, "url": ""}],
            "info": {
                "contributor": "",
                "date_created": "",
                "description": "Boulder detection results",
                "url": "",
                "version": "",
                "year": "",
            },
            "categories": [
                {"id": i + 1, "name": name, "supercategory": "none"}
                for i, name in enumerate(class_names)
            ],
            "images": [],
            "annotations": [],
        }

def poly_area(seg):
    if len(seg) >= 6:
        x = np.asarray(seg[0::2])
        y = np.asarray(seg[1::2])
        return float(0.5 * np.abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1))))
    return 0.0

def mask_to_seg(mask, shift_xy):
    mask = mask.astype(np.uint8)
    contours, _ = cv2.findContours(mask, cv2.RETR_LIST, cv2.CHAIN_APPROX_NONE)
    if not contours:
        return []
    # Use the largest contour to avoid tiny fragments.
    contour = max(contours, key=cv2.contourArea)
    boundary = np.reshape(contour, (1, -1))[0]
    boundary[0::2] = boundary[0::2] + shift_xy[0]
    boundary[1::2] = boundary[1::2] + shift_xy[1]
    return boundary.astype(float).tolist()

def ensure_rgb_uint8(arr):
    """Convert greyscale/RGBA/RGB arrays to RGB uint8."""
    if arr.ndim == 2:
        arr = np.stack([arr, arr, arr], axis=-1)
    if arr.shape[2] >= 4:
        arr = arr[:, :, :3]
    if arr.dtype != np.uint8:
        arr = arr.astype(np.float32)
        vmin, vmax = np.nanmin(arr), np.nanmax(arr)
        if vmax > vmin:
            arr = (arr - vmin) / (vmax - vmin) * 255.0
        arr = np.clip(arr, 0, 255).astype(np.uint8)
    return arr

def open_image_as_array(path):
    ext = Path(path).suffix.lower()
    if ext in [".tif", ".tiff"]:
        with TiffFile(path) as tif:
            arr = tif.asarray(out="memmap")
        return arr
    else:
        with Image.open(path) as im:
            return np.asarray(im.convert("RGB"))

def make_starts(length, window, step):
    if length <= window:
        return [0]
    starts = list(range(0, length - window + 1, step))
    last = length - window
    if starts[-1] != last:
        starts.append(last)
    return starts

def find_images(root_dir):
    paths = []
    for ext in IMAGE_EXTENSIONS:
        paths.extend(glob.glob(os.path.join(root_dir, "**", f"*{ext}"), recursive=True))
    return sorted(set(paths))

print("Utility functions loaded.")

## 5. Run detection

In [ ]:
pix_window = [WINDOW_HEIGHT, WINDOW_WIDTH]
pix_step = [
    max(1, int(round(WINDOW_HEIGHT * STEP_RATE_Y))),
    max(1, int(round(WINDOW_WIDTH * STEP_RATE_X))),
]

print("Window size:", pix_window)
print("Step size:", pix_step)
print("Output directory:", OUTPUT_DIR)

tag_list = find_images(DATA_DIR)
if MAX_IMAGES is not None:
    tag_list = tag_list[:MAX_IMAGES]

print("Number of images:", len(tag_list))
if len(tag_list) == 0:
    raise FileNotFoundError("No .tif/.tiff/.jpg/.jpeg images were found in the extracted ZIP.")

for image_id, image_path in enumerate(tqdm(tag_list, desc="Images"), start=1):
    image_name = Path(image_path).name
    image_stem = Path(image_path).stem
    print(f"Processing: {image_name}")

    arr = open_image_as_array(image_path)
    height, width = arr.shape[:2]

    row_starts = make_starts(height, WINDOW_HEIGHT, pix_step[0])
    col_starts = make_starts(width, WINDOW_WIDTH, pix_step[1])

    image_grid_data = []
    for y0 in row_starts:
        for x0 in col_starts:
            y1 = min(y0 + WINDOW_HEIGHT, height)
            x1 = min(x0 + WINDOW_WIDTH, width)
            image_grid_data.append([y0, y1, x0, x1])

    coco = CocoJsonInitialize(CLASS_NAMES)
    coco.jData["images"].append({
        "id": 1,
        "width": int(width),
        "height": int(height),
        "file_name": image_name,
        "license": 0,
        "flickr_url": "",
        "coco_url": "",
        "date_capture": "",
        "window_size": pix_window,
        "window_step_rate": [STEP_RATE_Y, STEP_RATE_X],
        "image_grid": image_grid_data,
    })

    ann_id = 0
    for y0 in tqdm(row_starts, desc="Rows", leave=False):
        for x0 in tqdm(col_starts, desc="Cols", leave=False):
            y1 = min(y0 + WINDOW_HEIGHT, height)
            x1 = min(x0 + WINDOW_WIDTH, width)

            tile_rgb = np.asarray(arr[y0:y1, x0:x1])
            tile_rgb = ensure_rgb_uint8(tile_rgb)

            if APPLY_SHARPENING:
                kernel = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])
                tile_rgb = cv2.filter2D(tile_rgb, -1, kernel)

            # Detectron2 expects BGR images.
            tile_bgr = tile_rgb[:, :, ::-1]
            outputs = predictor(tile_bgr)
            instances = outputs["instances"].to("cpu")

            if not instances.has("pred_masks") or len(instances) == 0:
                continue

            bboxes = instances.pred_boxes.tensor.numpy()
            scores = instances.scores.numpy()
            classes = instances.pred_classes.numpy()
            masks = instances.pred_masks.numpy()

            for idx in range(len(bboxes)):
                boundary = mask_to_seg(masks[idx], [x0, y0])
                if len(boundary) < 6:
                    continue

                ann_id += 1
                x_min, y_min, x_max, y_max = bboxes[idx]
                bbox = [
                    float(x_min + x0),
                    float(y_min + y0),
                    float(x_max - x_min + 1),
                    float(y_max - y_min + 1),
                ]

                coco.jData["annotations"].append({
                    "id": ann_id,
                    "image_id": 1,
                    "category_id": int(classes[idx]) + 1,
                    "segmentation": [boundary],
                    "area": poly_area(boundary),
                    "bbox": bbox,
                    "iscrowd": 0,
                    "attributes": {"occluded": 0},
                    "score": float(scores[idx]),
                })

            if SAVE_ANNOTATED_TILES:
                v = Visualizer(tile_rgb, metadata=detector_metadata, scale=1.0)
                out = v.draw_instance_predictions(instances)
                annotated_rgb = out.get_image()
                tile_name = f"{image_stem}_y{y0}_x{x0}_labeled.jpg"
                cv2.imwrite(os.path.join(OUTPUT_DIR, tile_name), annotated_rgb[:, :, ::-1])

    out_json = os.path.join(OUTPUT_DIR, f"{image_stem}_detect_object.json")
    with open(out_json, "w") as fp:
        json.dump(coco.jData, fp)
    print(f"Saved: {out_json}  annotations: {ann_id}")

print("Done.")

## 6. Download outputs

In [ ]:
import shutil
from google.colab import files

zip_base = f"{PROJECT_DIR}/boulder_detection_outputs"
zip_file = shutil.make_archive(zip_base, "zip", OUTPUT_DIR)
print(f"Created: {zip_file}")
files.download(zip_file)